# Implicit entity networks in 10 minutes

This notebook builds a (contextual) implicit entity network from the bundled example corpus,
explores it, ranks entities and sentences with the LOAD/EVELIN query model and clusters
edge contexts. It runs fully offline with the `GazetteerEntityExtractor`; swap in
`SpacyEntityExtractor` or `GLiNEREntityExtractor` for real named-entity recognition.

In [1]:
from implicit_word_network import (
    ContextualEdgeClusterer,
    GazetteerEntityExtractor,
    ImplicitNetworkPipeline,
    load_example_corpus,
)

corpus = load_example_corpus()
print(corpus)
print(corpus[0].text[:200], "...")

Corpus(documents=6, meta={})
Richard Phillips Feynman was an American theoretical physicist, known for his work in the path integral formulation of quantum mechanics, the theory of quantum electrodynamics, the physics of the supe ...


## 1. Extract entities

A gazetteer is the simplest extractor: a dictionary of surface forms per entity type.

In [2]:
gazetteer = {
    "PERSON": [
        "Feynman",
        "Richard Feynman",
        "Schwinger",
        "Tomonaga",
        "Einstein",
        "Bohr",
        "Dirac",
        "Heisenberg",
        "Alfred Nobel",
        "Curie",
        "Planck",
        "Pauli",
    ],
    "ORG": [
        "Caltech",
        "California Institute of Technology",
        "Nobel Prize",
        "Rogers Commission",
        "Manhattan Project",
        "Los Alamos",
        "Nobel Foundation",
        "Princeton",
        "Cambridge",
    ],
    "LOC": [
        "Copenhagen",
        "Berlin",
        "Germany",
        "Denmark",
        "Stockholm",
        "Sweden",
        "Paris",
        "United States",
        "Zurich",
        "Vienna",
        "Munich",
        "New York",
    ],
}
extractor = GazetteerEntityExtractor(gazetteer)
doc = extractor.annotate_text(corpus[0].text)
print(doc)
[(m.text, m.label, m.sentence) for m in doc.mentions[:8]]

AnnotatedDocument(id=0, sentences=12, tokens=333, mentions=16)


[('Feynman', 'PERSON', 0),
 ('Feynman', 'PERSON', 1),
 ('Nobel Prize', 'ORG', 1),
 ('Schwinger', 'PERSON', 1),
 ('Tomonaga', 'PERSON', 1),
 ('Feynman', 'PERSON', 1),
 ('Feynman', 'PERSON', 1),
 ('Feynman', 'PERSON', 2)]

## 2. Build the network

`window=2` means two mentions cooccur when they are at most two sentences apart; the
edge weight is $\omega = \sum \exp(-\delta)$ over all cooccurrences.

In [3]:
pipeline = ImplicitNetworkPipeline(extractor, window=2)
network = pipeline.run(corpus)
print(network.summary())

ImplicitNetwork
  documents : 6
  sentences : 33
  entities  : 13
  terms     : 360
  mentions  : 33
  edges     : 19
  window    : 2 (decay=exponential)


In [4]:
for edge in network.edges(top_k=8):
    print(
        f"{edge.weight:6.2f}  {edge.source.text} [{edge.source.label}] -- {edge.target.text} [{edge.target.label}]  (n={edge.count})"
    )

  3.74  Nobel Prize [ORG] -- Alfred Nobel [PERSON]  (n=5)
  3.74  Feynman [PERSON] -- Nobel Prize [ORG]  (n=5)
  3.74  Feynman [PERSON] -- Tomonaga [PERSON]  (n=5)
  3.74  Feynman [PERSON] -- Schwinger [PERSON]  (n=5)
  2.64  Feynman [PERSON] -- California Institute of Technology [ORG]  (n=5)
  1.00  Schwinger [PERSON] -- Tomonaga [PERSON]  (n=1)
  1.00  Planck [PERSON] -- Einstein [PERSON]  (n=1)
  1.00  Nobel Prize [ORG] -- Tomonaga [PERSON]  (n=1)


## 3. Explore an entity

Neighbours, characteristic terms and the sentences an entity appears in.

In [5]:
feynman = network.entity("Feynman", "PERSON")
print("neighbours:", [(n.text, round(w, 2)) for n, w in network.neighbors(feynman, k=5)])
print(
    "LOAD-weighted:",
    [(n.text, round(w, 2)) for n, w in network.neighbors(feynman, k=5, weighting="load")],
)
print("terms:", [(t.text, c) for t, c in network.entity_terms(feynman, k=8)])
print("mentioned in", len(network.sentences_of(feynman)), "sentences")

neighbours: [('Nobel Prize', 3.74), ('Schwinger', 3.74), ('Tomonaga', 3.74), ('California Institute of Technology', 2.64), ('Rogers Commission', 0.5)]
LOAD-weighted: [('Schwinger', 3.17), ('Tomonaga', 3.17), ('Nobel Prize', 1.07), ('California Institute of Technology', 0.76), ('Rogers Commission', 0.14)]
terms: [('physics', 12), ('quantum', 6), ('lectures', 6), ('known', 5), ('became', 5), ('theoretical', 4), ('electrodynamics', 4), ('work', 3)]
mentioned in 7 sentences


## 4. Provenance: where do two entities cooccur?

In [6]:
for cooc, context in zip(
    network.cooccurrences(feynman, ("Nobel Prize", "ORG")),
    network.contexts(feynman, ("Nobel Prize", "ORG")),
):
    print(f"delta={cooc.delta} weight={cooc.weight:.2f} doc={cooc.document}: {context[:110]}...")

delta=1 weight=0.37 doc=0: Richard Phillips Feynman was an American theoretical physicist, known for his work in the path integral formul...
delta=0 weight=1.00 doc=0: For contributions to the development of quantum electrodynamics, Feynman received the Nobel Prize in Physics i...
delta=0 weight=1.00 doc=0: For contributions to the development of quantum electrodynamics, Feynman received the Nobel Prize in Physics i...
delta=0 weight=1.00 doc=0: For contributions to the development of quantum electrodynamics, Feynman received the Nobel Prize in Physics i...
delta=1 weight=0.37 doc=0: For contributions to the development of quantum electrodynamics, Feynman received the Nobel Prize in Physics i...


## 5. Ranking queries (LOAD / EVELIN)

Which entities, sentences and documents matter for a set of query entities?

In [7]:
query = [feynman, ("Nobel Prize", "ORG")]
print("entities:", [(n.text, round(s, 2)) for n, s in network.rank_entities(query, k=5)])
for sentence, score in network.rank_sentences(query, k=3):
    print(f"{score:.2f}  [doc {sentence.document}] {sentence.text[:120]}")
print("documents:", [(d.id, round(s, 2)) for d, s in network.rank_documents(query, k=3)])

entities: [('Schwinger', 2.0), ('Tomonaga', 2.0), ('Alfred Nobel', 0.56), ('Nobel Foundation', 0.29), ('California Institute of Technology', 0.2)]
2.37  [doc 0] For contributions to the development of quantum electrodynamics, Feynman received the Nobel Prize in Physics in 1965 joi
1.47  [doc 1] It is one of the five Nobel Prizes established by the will of Alfred Nobel in 1895 and awarded since 1901; the others be
1.32  [doc 0] Richard Phillips Feynman was an American theoretical physicist, known for his work in the path integral formulation of q
documents: [(0, 3.0), (1, 1.96)]


## 6. Contextual edges (CIEN)

Cluster the contexts of an edge to separate the different relations two entities have.
The bag-of-words embedder is used here; `SentenceTransformerEmbedder` gives neural contexts.

In [8]:
clusterer = ContextualEdgeClusterer(eps=0.6)
for cluster in clusterer.cluster_edge(network, feynman, ("Nobel Prize", "ORG")):
    print(f"cluster {cluster.label}: {cluster.size} cooccurrence(s), weight {cluster.weight:.2f}")
    print("   ", cluster.contexts[0][:100], "...")

cluster 0: 5 cooccurrence(s), weight 3.74
    Richard Phillips Feynman was an American theoretical physicist, known for his work in the path integ ...


## 7. Export

Continue in NetworkX, pandas or Gephi (GraphML), or persist the full network.

In [9]:
graph = network.to_networkx(min_weight=0.5)
print(graph)
data = network.to_dict(top_k=3)
data["edges"]

Graph with 13 nodes and 11 edges


[{'source': 'feynman|PERSON',
  'target': 'nobel prize|ORG',
  'weight': 3.7357588823428847,
  'count': 5},
 {'source': 'feynman|PERSON',
  'target': 'schwinger|PERSON',
  'weight': 3.7357588823428847,
  'count': 5},
 {'source': 'feynman|PERSON',
  'target': 'tomonaga|PERSON',
  'weight': 3.7357588823428847,
  'count': 5}]